In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

%matplotlib inline


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/.venv/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/.venv/lib/python3.11/site-packages/traitlets/config/application.py", line 1080, in launch_instance
    app.start()
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/

In [2]:
words = open('names.txt', 'r').read().splitlines()
words[:8], len(words)

(['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia'],
 32033)

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [4]:
print(chars)
print(stoi)
print(itos)

['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
X = []
Y = []

block_size = 3

for w in words:
    context = [0] * block_size

    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

In [6]:
X = torch.tensor(X)
Y = torch.tensor(Y)

In [7]:
# print(X)
# print(Y)
print(X.shape)
print(Y.shape)

torch.Size([228146, 3])
torch.Size([228146])


In [8]:
C = torch.randn(27,10)
C.shape
print(X[0])
print(X[0].shape)

emb = C[X[0]]

print(emb.shape)

tensor([0, 0, 0])
torch.Size([3])
torch.Size([3, 10])


In [9]:
W1 = torch.randn(30, 200)
b1 = torch.randn(200)

W2 = torch.randn((200, 27))
b2 = torch.randn(27)

C = torch.randn(27,10)

parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad = True

In [10]:
emb = C[X[2]]
emb_flat = emb.view(-1, 30)

z = emb_flat @ W1
print("z:", z.shape)
z = z + b1
print("z after bias:", z.shape)
h = torch.tanh(z)
print("h:", h.shape)
logits = h @ W2 + b2
logits.shape, logits

z: torch.Size([1, 200])
z after bias: torch.Size([1, 200])
h: torch.Size([1, 200])


(torch.Size([1, 27]),
 tensor([[ 34.4214,  -6.8628,   6.2022,  24.7001, -18.3839,  -2.6148, -11.9498,
           -9.1289,   5.8496,  12.8853, -10.3245,   6.7565,  -0.9273,  -9.9329,
           24.8126,   4.9763,   2.2401,  -4.5973, -11.1938,   4.5308,   0.1572,
           11.4155,  23.0927,  -8.4697, -14.2558,  -7.7789,   9.4237]],
        grad_fn=<AddBackward0>))

In [11]:
counts = logits.exp()
counts.shape, counts

(torch.Size([1, 27]),
 tensor([[8.8923e+14, 1.0460e-03, 4.9385e+02, 5.3346e+10, 1.0375e-08, 7.3181e-02,
          6.4603e-06, 1.0849e-04, 3.4710e+02, 3.9446e+05, 3.2820e-05, 8.5964e+02,
          3.9562e-01, 4.8550e-05, 5.9701e+10, 1.4493e+02, 9.3945e+00, 1.0079e-02,
          1.3759e-05, 9.2837e+01, 1.1703e+00, 9.0713e+04, 1.0691e+10, 2.0974e-04,
          6.4388e-07, 4.1848e-04, 1.2379e+04]], grad_fn=<ExpBackward0>))

In [12]:
prob = counts / counts.sum(1, keepdims=True)  # why do we need, keepdims here
prob.shape, prob

(torch.Size([1, 27]),
 tensor([[9.9986e-01, 1.1761e-18, 5.5530e-13, 5.9984e-05, 1.1666e-23, 8.2286e-17,
          7.2640e-21, 1.2198e-19, 3.9028e-13, 4.4354e-10, 3.6903e-20, 9.6659e-13,
          4.4485e-16, 5.4590e-20, 6.7129e-05, 1.6296e-13, 1.0563e-14, 1.1333e-17,
          1.5471e-20, 1.0439e-13, 1.3159e-15, 1.0200e-10, 1.2022e-05, 2.3583e-19,
          7.2399e-22, 4.7054e-19, 1.3919e-11]], grad_fn=<DivBackward0>))

In [13]:

target = Y[2]

loss_manual = -prob[0, target].log()

loss_pytorch = F.cross_entropy(
    logits,
    target.unsqueeze(0)
)

print("manual:", loss_manual.item())
print("PyTorch:", loss_pytorch.item())

manual: 44.354434967041016
PyTorch: 44.354434967041016


In [14]:

loss = F.cross_entropy(logits, target.unsqueeze(0))
loss

tensor(44.3544, grad_fn=<NllLossBackward0>)

In [15]:
loss.backward()

In [16]:
print(W2.grad.shape)
print(W1.grad.shape)
print(b2.grad.shape)
print(b1.grad.shape)
print(C.grad.shape)

torch.Size([200, 27])
torch.Size([30, 200])
torch.Size([27])
torch.Size([200])
torch.Size([27, 10])


In [17]:
print(W2.grad[0, 0])
print(W2[0, 0])

tensor(-0.9233)
tensor(-0.9127, grad_fn=<SelectBackward0>)


In [18]:
idx = (0, 0)
old = W2[idx].item()
old

-0.9127376675605774

In [19]:
eps = 1

W2.data[idx] = old + eps

# recompute forward pass
emb = C[X[2]]
h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
logits_plus = h @ W2 + b2
loss_plus = F.cross_entropy(logits_plus, target.unsqueeze(0))

W2.data[idx] = old

In [20]:
numerical_grad = (loss_plus.item() - loss.item())/eps

print("autograd:", W2.grad[idx].item())
print("numerical:", numerical_grad)

autograd: -0.9232742190361023
numerical: -0.9231910705566406


In [22]:
n1 = int(0.8 * len(X))
n2 = int(0.9 * len(X))

Xtr, Ytr = X[:n1], Y[:n1]
Xdev, Ydev = X[n1:n2], Y[n1:n2]
Xte, Yte = X[n2:], Y[n2:]

Xtr.shape, Ytr.shape

(torch.Size([182516, 3]), torch.Size([182516]))

In [38]:
ix = torch.randint(0, Xtr.shape[0], (32,))

In [39]:
ix

tensor([ 59514,  22462,  41164, 135935,  80164, 146423,   9643,  76698, 125018,
        121150, 168781,   4360, 150713, 160506, 133057,  18668,  79312,  22161,
         66200, 149916,  35403,   8075, 122859,  81403, 159637, 165839, 171902,
        163139,  97538,   1440, 163565, 168545])

In [41]:
Xtr[ix].shape , Ytr[ix].shape

(torch.Size([32, 3]), torch.Size([32]))

In [42]:
emb = C[Xtr[ix]]
emb.view(-1, 30).shape

torch.Size([32, 30])

In [ ]:
## Starting form SCRATCH

g = torch.Generator().manual_seed(2147483647)

C = torch.randn((27, 10), generator=g)
W1 = torch.randn((30, 200), generator=g)
b1 = torch.randn(200, generator=g)

W2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)

parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad = True

In [44]:
for i in range(10):

    ix = torch.randint(0, Xtr.shape[0], (32,))

    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
    logits = h @ W2 + b2

    loss = F.cross_entropy(logits, Ytr[ix])

    for p in parameters:
        p.grad = None

    loss.backward()

    lr = 0.1

    for p in parameters:
        p.data += -lr * p.grad

    print(i, loss.item())

0 21.80514144897461
1 27.308475494384766
2 23.593273162841797
3 21.876235961914062
4 23.801044464111328
5 20.926441192626953
6 18.07098960876465
7 18.167171478271484
8 15.133831977844238
9 18.121328353881836
